In [1]:
import sys
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import openpyxl

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Environment ready.")

Python: 3.11.8 (main, Feb 19 2026, 01:12:21) [Clang 17.0.0 (clang-1700.6.3.2)]
pandas: 3.0.3
NumPy: 2.4.6
scikit-learn: 1.9.0
Environment ready.


## Raw Dataset Inspection

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "emails.csv"

print("Interpreter:", Path(sys.executable).name)
print("Dataset:", RAW_DATA.relative_to(PROJECT_ROOT))
print("Exists:", RAW_DATA.exists())
print("Size (GB):", round(RAW_DATA.stat().st_size / 1024**3, 2))

Interpreter: python
Dataset: data/raw/emails.csv
Exists: True
Size (GB): 1.33


### Preview the source data

In [3]:
import pandas as pd

preview = pd.read_csv(RAW_DATA, nrows=5)

print("Columns:", preview.columns.tolist())
print("Preview shape:", preview.shape)
display(preview)

Columns: ['file', 'message']
Preview shape: (5, 2)


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


## Full Dataset Inventory

The raw CSV is processed in chunks to verify its size and basic quality without loading the complete dataset into memory.

In [4]:
CHUNK_SIZE = 10_000

total_rows = 0
missing_file = 0
missing_message = 0
blank_message = 0

for chunk in pd.read_csv(
    RAW_DATA,
    usecols=["file", "message"],
    chunksize=CHUNK_SIZE
):
    total_rows += len(chunk)
    missing_file += chunk["file"].isna().sum()
    missing_message += chunk["message"].isna().sum()

    blank_message += (
        chunk["message"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )

inventory = {
    "total_rows": int(total_rows),
    "missing_file": int(missing_file),
    "missing_message": int(missing_message),
    "blank_message": int(blank_message),
}

inventory

{'total_rows': 517401,
 'missing_file': 0,
 'missing_message': 0,
 'blank_message': 0}

### Inventory interpretation

The raw Enron CSV contains **517,401 records**. The inspection found
**0** missing file identifiers, **0** missing messages, and
**0** blank messages. The dataset therefore requires parsing,
deduplication, and usability checks before sampling or modeling.